In [0]:
# MAGIC %md
# ============================================================
# NOTEBOOK   : gold_payment_analysis
# PURPOSE    : Payment type and instalment analysis
# SOURCE     : silver.order_payments, silver.orders
# DESTINATION: fincompliance_catalog.gold.payment_analysis
# METRICS:
#   - Revenue by payment type
#   - Instalment usage by payment type
#   - Average instalment value
#   - Payment value range distribution
# ============================================================

In [0]:
%run ../configs/config

In [0]:
authenticate_spark(spark)
set_catalog(spark)

In [0]:
from pyspark.sql.functions import (
    col,
    count,
    sum as spark_sum,
    avg,
    round as spark_round,
    when,
    current_timestamp
)

In [0]:
# ============================================================
# CELL 5 - READ FROM SILVER
# ============================================================

df_payments = spark.table(f"{SILVER_DB}.order_payments")
df_orders = spark.table(f"{SILVER_DB}.orders")

print("=" * 60)
print("SILVER TABLES LOADED")
print("=" * 60)
print(f"order_payments : {df_payments.count():,} rows")
print(f"orders         : {df_orders.count():,} rows")

print("\nPayment type breakdown:")
df_payments \
    .groupBy("payment_type") \
    .count() \
    .orderBy("count", ascending=False) \
    .show(truncate=False)

print("\nInstalment breakdown:")
df_payments \
    .groupBy("is_instalment") \
    .count() \
    .orderBy("is_instalment") \
    .show(truncate=False)

In [0]:
# ============================================================
# CALCULATE PAYMENT ANALYSIS BY TYPE AND YEAR
# Join payments with delivered orders
# Group by payment type and year
# ============================================================

df_payment_analysis = (
    df_payments
    .join(
        df_orders.filter(col("order_status") == "delivered")
                  .select("order_id", "order_year"),
        on="order_id",
        how="inner"
    )
    .groupBy("payment_type", "order_year")
    .agg(
        count("order_id").alias("total_transactions"),
        spark_round(spark_sum("payment_value"), 2)
            .alias("total_revenue"),
        spark_round(avg("payment_value"), 2)
            .alias("avg_payment_value"),
        spark_round(avg("payment_installments"), 1)
            .alias("avg_installments"),
        spark_round(avg("instalment_value"), 2)
            .alias("avg_instalment_value"),
        spark_sum(
            when(col("is_instalment") == 1, 1)
            .otherwise(0)
        ).alias("instalment_count"),
        spark_sum(
            when(col("is_instalment") == 0, 1)
            .otherwise(0)
        ).alias("single_payment_count")
    )
    .withColumn(
        "instalment_rate_pct",
        spark_round(
            (col("instalment_count") /
             col("total_transactions")) * 100, 2
        )
    )
    .orderBy("order_year", "total_revenue",
             ascending=[True, False])
)

print("Payment analysis by type and year:")
df_payment_analysis.show(20, truncate=False)

print(f"\nTotal rows : {df_payment_analysis.count()}")

In [0]:
# ============================================================
# ADD GOLD AUDIT COLUMN
# ============================================================

df_payment_analysis = df_payment_analysis \
    .withColumn("gold_updated_at", current_timestamp())

print("Final columns:")
for col_name in df_payment_analysis.columns:
    print(f"  {col_name}")
print(f"Total rows : {df_payment_analysis.count()}")

In [0]:
# ============================================================
# WRITE TO GOLD
# ============================================================

(
    df_payment_analysis.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{GOLD_DB}.payment_analysis")
)

print(f"Written to {GOLD_DB}.payment_analysis")
print(f"Rows : {df_payment_analysis.count()}")

In [0]:
# ============================================================
# VERIFICATION
# ============================================================
print("=" * 60)
print("GOLD.PAYMENT_ANALYSIS VERIFICATION")
print("=" * 60)

df_gold = spark.table(f"{GOLD_DB}.payment_analysis")
print(f"Total rows    : {df_gold.count()}")
print(f"Total columns : {len(df_gold.columns)}")

print("\nTotal revenue by payment type:")
df_gold \
    .groupBy("payment_type") \
    .agg(
        spark_round(spark_sum("total_revenue"), 2)
            .alias("total_revenue"),
        spark_sum("total_transactions")
            .alias("total_transactions")
    ) \
    .orderBy(col("total_revenue").desc()) \
    .show(truncate=False)

print("\nInstalment rate trend by year:")
df_gold \
    .filter(col("payment_type") == "credit_card") \
    .select(
        "order_year",
        "total_transactions",
        "instalment_rate_pct",
        "avg_installments",
        "avg_instalment_value"
    ) \
    .orderBy("order_year") \
    .show(truncate=False)

print("=" * 60)
print("gold.payment_analysis verification complete!")
print("=" * 60)